In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# ------------------------------
# 1. Load data
# ------------------------------
df = pd.read_csv("autofocus_training_final 1.3_2500.csv")

# Feature columns – adjust if you want to include X,Y
feature_cols = ['Z_prev', 'variance_ratio', 'Variance_sq',
                'is_improving', 'direction', 'dVariance_abs']
target_col = 'target_step'

X = df[feature_cols]
y = df[target_col]

print(f"Loaded {len(df)} samples.")

# ------------------------------
# 2. Train / test split (80/20)
# ------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ------------------------------
# 3. Scale features
# ------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ------------------------------
# 4. Train Random Forest
# ------------------------------
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)

# ------------------------------
# 5. Predict on test set & compute metrics
# ------------------------------
y_pred = model.predict(X_test_scaled)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("\n================= Test Set Metrics =================")
print(f"MAE    : {mae:.2f} steps")
print(f"RMSE   : {rmse:.2f} steps")
print(f"R²     : {r2:.4f}")
print("====================================================")

# ------------------------------
# 6. Cross-validation (more robust)
# ------------------------------
cv_scores = cross_val_score(model, X_train_scaled, y_train,
                            cv=5, scoring='r2', n_jobs=-1)
print(f"\n5‑fold CV R² scores: {cv_scores}")
print(f"Mean CV R²: {cv_scores.mean():.4f}")

# ------------------------------
# 7. Save model and scaler
# ------------------------------
joblib.dump(model, 'new_rf_autofocusv55_model.pkl')
joblib.dump(scaler, 'new_microscopev55_scaler.pkl')
print("\nModel and scaler saved to disk.")

Loaded 1040 samples.

================= Test Set Metrics =================
MAE    : 137.44 steps
RMSE   : 228.16 steps
R²     : 0.9814

5‑fold CV R² scores: [0.98634623 0.98552385 0.97644137 0.98365161 0.98621796]
Mean CV R²: 0.9836

Model and scaler saved to disk.
